# Produce a score

# 🧾 Summary of the Scoring Notebook

## 🧠 1. Quality Dimensions

The evaluation framework is based on 7 dimensions:

### 1. Faithfulness
Measures factual consistency with the source article (absence of hallucinations)

### 2. Relevance
Semantic similarity between article and summary

### 3. Salience
Coverage of important information via named entities

### 4. Conciseness
Degree of compression relative to the source article

### 5. Truncation
Detection of incomplete or cut-off sentences

### 6. Noise
Presence of non-informative content (metadata, labels, captions)

---

## ⚙️ 2. Methodology

### 🔹 Semantic similarity
- model: `paraphrase-multilingual-mpnet-base-v2`
- used for relevance and redundancy estimation

### 🔹 Japanese processing
- tokenization using SudachiPy
- heuristic extraction of entities (kanji-based patterns)

---

## 🧮 3. Score Construction

### Step 1 — Feature extraction
Each summary is evaluated using:
- article-summary semantic similarity
- hallucination proxy score
- entity coverage
- compression ratio
- noise and truncation detection

---

### Step 2 — Normalization

Raw metrics are transformed into interpretable scores:

- faithfulness = 1 - hallucination_score
- salience = entity recall
- conciseness = distribution-based normalization of compression ratio
- binary flags converted into penalties

---

### Step 3 — Global scoring

Final weighted score:

- Faithfulness: 30%
- Salience: 20%
- Relevance: 15%
- Conciseness: 10%
- Truncation: 10%
- Noise: 5%

---

## 🏁 4. Outputs

Each summary is assigned:
- a global score (0–1)
- per-dimension scores
- a qualitative label:
  - excellent
  - good
  - medium
  - poor

---

## 💡 5. Conclusion

This notebook implements a **multi-dimensional evaluation framework for LLM-generated summaries**, combining:

- semantic similarity modeling
- Japanese-specific NLP heuristics
- entity-based coverage analysis
- statistical normalization techniques

The goal is to produce **interpretable and comparable summary quality scores**, while explicitly acknowledging the limitations of heuristic-based evaluation methods.

In [10]:
import numpy as np
import json
from collections import defaultdict
from sentence_transformers import SentenceTransformer, util
from sudachipy import dictionary, tokenizer
import re

# ----------------------------
# Load data
# ----------------------------
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

articles = load_jsonl("data/articles.jsonl")
summaries = load_jsonl("data/summaries.jsonl")

article_map = {a["article_id"]: a for a in articles}

# ----------------------------
# Model
# ----------------------------
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

# ----------------------------
# Tokenizer (JP)
# ----------------------------
tokenizer_obj = dictionary.Dictionary().create()
mode = tokenizer.Tokenizer.SplitMode.C

def tokenize(text):
    return [m.surface() for m in tokenizer_obj.tokenize(text, mode)]

# ----------------------------
# Sentence split
# ----------------------------
def split_sentences(text):
    return re.split(r"[。！？\n]", text)

# ----------------------------
# Entities
# ----------------------------
def extract_entities(text):
    return set(re.findall(r"[一-龯]{2,}", text))

# ----------------------------
# Embedding similarity
# ----------------------------
def sim(a, b):
    emb = model.encode([a, b], normalize_embeddings=True)
    return float(util.cos_sim(emb[0], emb[1]))

# ----------------------------
# Irrelevance
# ----------------------------
def relevance_score(article, summary):
    return sim(article, summary)

# ----------------------------
# Hallucination
# ----------------------------
def hallucination_score(article, summary):
    article_tokens = set(tokenize(article))
    summary_tokens = set(tokenize(summary))

    lexical_overlap = len(summary_tokens & article_tokens) / (len(summary_tokens) + 1e-6)

    article_ent = extract_entities(article)
    summary_ent = extract_entities(summary)

    new_entities_ratio = len(summary_ent - article_ent) / (len(summary_ent) + 1e-6)

    # combine (bounded, smoother than before)
    return 0.6 * (1 - lexical_overlap) + 0.4 * new_entities_ratio

# ----------------------------
# Truncation
# ----------------------------
def truncation_score(summary):
    return 1.0 if summary.strip().endswith(("。", "！", "？")) else 0.0

# ----------------------------
# Salience
# ----------------------------
def salience_score(article, summary):
    a_ent = extract_entities(article)
    s_ent = extract_entities(summary)

    if not a_ent:
        return 0.5

    return len(a_ent & s_ent) / len(a_ent)

# ----------------------------
# Compression
# ----------------------------
def compression_ratio(article, summary):
    return len(tokenize(summary)) / (len(tokenize(article)) + 1e-6)

# ----------------------------
# Noise
# ----------------------------
def noise_score(summary):
    patterns = [r"\(.*?\)", r"©", r"写真", r"記者", r"報道"]
    return any(re.search(p, summary) for p in patterns)

# ----------------------------
# Group
# ----------------------------
grouped = defaultdict(list)
for s in summaries:
    grouped[s["article_id"]].append(s)

# collect compression ratios for normalization
all_ratios = []

for article_id, summary_list in grouped.items():
    article = article_map[article_id]["text"]

    for s in summary_list:
        all_ratios.append(compression_ratio(article, s["summary"]))

p25, p50, p75 = np.percentile(all_ratios, [25, 50, 75])

def compression_score(ratio):
    if ratio < p25:
        return 0.7
    elif ratio < p75:
        return 1.0
    else:
        return 0.5

# ----------------------------
# Evaluation
# ----------------------------
results = []

for article_id, summary_list in grouped.items():
    article = article_map[article_id]["text"]
    texts = [s["summary"] for s in summary_list]

    for s in summary_list:
        summary = s["summary"]

        rel = relevance_score(article, summary)
        hall = hallucination_score(article, summary)
        sal = salience_score(article, summary)
        ratio = compression_ratio(article, summary)

        results.append({
            "summary_id": s["summary_id"],
            "article_id": article_id,

            "relevance": rel,
            "hallucination": hall,
            "salience": sal,

            "compression_ratio": ratio,
            "compression_score": compression_score(ratio),

            "truncation": truncation_score(summary),
            "noise": noise_score(summary)
        })

# ----------------------------
# Normalization
# ----------------------------
def normalize(r):
    return {
        "faithfulness": 1 - r["hallucination"],
        "relevance": r["relevance"],
        "salience": r["salience"],
        "conciseness": r["compression_score"],
        "no_truncation": r["truncation"],
        "no_noise": 1 - int(r["noise"])
    }

WEIGHTS = {
    "faithfulness": 0.30,
    "relevance": 0.15,
    "salience": 0.20,
    "conciseness": 0.10,
    "no_truncation": 0.10,
    "no_noise": 0.05
}

def score(norm):
    return sum(norm[k] * w for k, w in WEIGHTS.items())

# ----------------------------
# Final output
# ----------------------------
final_results = []

for r in results:
    norm = normalize(r)
    r["global_score"] = score(norm)

    if r["global_score"] > 0.8:
        r["label"] = "excellent"
    elif r["global_score"] > 0.6:
        r["label"] = "good"
    elif r["global_score"] > 0.4:
        r["label"] = "medium"
    else:
        r["label"] = "poor"

    final_results.append(r)

with open("scores.jsonl", "w", encoding="utf-8") as f:
    for r in final_results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")